# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata_obj = dataset.metadata
metadata_json = metadata_obj.to_json()  # Get a dict representation for pretty printing
print(f"{metadata_json['name']}: {metadata_json['description']}")
print("\nOther metadata:")
pprint.pprint({
    'Identifier': metadata_json.get('identifier', ''),
    'Citation': metadata_json.get('citeAs', ''),
    'Authors': metadata_json.get('author', ''),
    'Published Date': metadata_json.get('datePublished', ''),
    'License': metadata_json.get('license', '')
})

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore record sets (tables) present in the dataset via their `@id` fields
record_sets = metadata_obj.record_sets
print(f"Found {len(record_sets)} record set(s) in the dataset.\n")

for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet @id: {rs.id}")
    print(f"    name: {rs.name}")
    print(f"    description: {getattr(rs, 'description', '')}")
    
    # List fields in this record set
    print("    Fields/Columns (@id : name : dataType):")
    for field in rs.fields:
        print(f"      - {field.id} : {field.name} : {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose record set(s) by @id for extraction
# You may edit this list to select the relevant tables/record sets for your analysis
record_set_ids = [rs.id for rs in metadata_obj.record_sets]

# We'll load all record sets into dataframes (by @id for easy reference)
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded: {rs_id} ({len(df)} rows, {len(df.columns)} columns)")
    except Exception as e:
        print(f"Could not load records for {rs_id}:", e)
        dataframes[rs_id] = None

# Show columns of the primary record set (usually the first)
main_rs_id = record_set_ids[0] if len(record_set_ids) else None
if main_rs_id and dataframes[main_rs_id] is not None:
    print(f"\nPrimary dataframe columns for record_set '{main_rs_id}':")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()
else:
    print("No records loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
# EDA: pick a record set and a numeric field by `@id`
record_set_id = main_rs_id
df = dataframes.get(record_set_id, None)
if df is not None and not df.empty:
    # Find a numeric field (float/int) by inspecting dtypes --- fallback to user selection
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        print("No numeric columns detected in dataframe. Please refer to the field list above and set 'numeric_field_id' manually.")
    else:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field '@id': {numeric_field_id}")
        
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()

        print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}")

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"First 5 values for normalized {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Choose a categorical/grouping field by inspecting object columns (could be chosen by user)
        object_cols = df.select_dtypes(include=['object']).columns.tolist()
        if object_cols:
            group_field_id = object_cols[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped (mean) {numeric_field_id} by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print("No categorical/text columns found for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and 'numeric_field_id' in locals():
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If can group - boxplot by group_field_id
    if 'group_field_id' in locals():
        plt.figure(figsize=(10, 5))
        top_groups = df[group_field_id].value_counts().index[:5]
        sns.boxplot(data=df[df[group_field_id].isin(top_groups)], x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (top 5 groups)")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.